# 📊 Logistic Regression & TF-IDF Ensemble - Smart MCQ Solver (With Advanced Wandb Logging)

### 📌 Overview & Wandb Tracking Features
This notebook implements the complete **Logistic Regression + Ridge + SGD TF-IDF Ensemble** with **Rich Fold-Wise and Metric-Wise Weights & Biases (Wandb) Logging**.

- **Primary Kaggle Metric**: `MAP@3` (Mean Average Precision at 3)
- **Secondary Evaluation Metrics**:
  - `Top-1 Accuracy`: Standard classification accuracy (single best prediction).
  - `Top-3 Accuracy`: Recall metric checking if the ground truth is within top 3 ranked answers.
  - `F1-Score (Macro)`: Harmonic mean of precision and recall.
  - `Precision (Macro)`: Proportion of correct positive predictions.
  - `Recall (Macro)`: Coverage of actual ground truth labels.
  - `Log Loss`: Binary Cross Entropy of option probability predictions.

- **Wandb Dashboard Logs**:
  - **Hyperparameter Configuration**: Track C, alpha, TF-IDF max_features, n-gram ranges, n_splits.
  - **Per-Fold Metric Progression**: Logs `fold/val_loss`, `fold/val_map3`, `fold/val_top1_accuracy`, `fold/val_top3_accuracy`, `fold/val_f1_score`, `fold/val_precision`, `fold/val_recall`.
  - **Comparative Model Summary Table**: Automatically generates a `wandb.Table` comparing Logistic Regression, Ridge Classifier, SGD Classifier, and Ensemble Blend.

---

In [5]:
# Cell 1: Core Imports & Dependencies
import os
import re
import warnings
import numpy as np
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from scipy.special import expit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, log_loss

warnings.filterwarnings('ignore')
print(f"Core libraries imported successfully! seed={SEED}")


Core libraries imported successfully! seed=42


In [6]:
# Cell 2: Weights & Biases (Wandb) Authentication & Setup
import os
try:
    import wandb
    WANDB_ENTITY  = "23f2001849-dl-genai-project"
    WANDB_PROJECT = "23f2001849-t22026"

    key = None
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception:
        key = os.environ.get("WANDB_API_KEY")

    wandb.login(key=key, relogin=True)
    USE_WANDB = True
    print("Wandb initialized successfully!")
except Exception as e:
    USE_WANDB = False
    print(f"Wandb running in offline fallback mode ({e})")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Wandb initialized successfully!


In [7]:
# Cell 3: Dataset Loading & Setup
def load_datasets():
    train_candidates = [
        "train.csv",
        "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",
        "/kaggle/input/smart-mcq-solver-challenge/train.csv"
    ]
    test_candidates = [
        "test.csv",
        "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv",
        "/kaggle/input/smart-mcq-solver-challenge/test.csv"
    ]
    
    train_path = next((p for p in train_candidates if os.path.exists(p)), train_candidates[0])
    test_path = next((p for p in test_candidates if os.path.exists(p)), test_candidates[0])
    
    df_train = pd.read_csv(train_path).fillna('')
    df_test = pd.read_csv(test_path).fillna('')
    
    return df_train, df_test

train, test = load_datasets()
OPTION_COLS = ['A', 'B', 'C', 'D', 'E']

print(f"Train shape: {train.shape}")
print(f"Test shape : {test.shape}")
train.head(3)


Train shape: (2000, 8)
Test shape : (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [8]:
# Cell 4: Comprehensive Metric Evaluation Functions
def apk(actual, predicted, k=3):
    if actual in predicted[:k]:
        return 1.0 / (predicted[:k].index(actual) + 1.0)
    return 0.0

def mapk(actuals, preds, k=3):
    return float(np.mean([apk(a, p, k) for a, p in zip(actuals, preds)]))

def compute_all_metrics(actuals, pred_lists):
    """
    Computes MAP@3, Top-1 Accuracy, Top-3 Recall Accuracy, F1-Score, Precision, and Recall.
    """
    map3_score = mapk(actuals, pred_lists)
    
    top1_preds = [p[0] for p in pred_lists]
    top1_acc = accuracy_score(actuals, top1_preds)
    
    top3_acc = np.mean([1 if a in p[:3] else 0 for a, p in zip(actuals, pred_lists)])
    
    prec, rec, f1, _ = precision_recall_fscore_support(actuals, top1_preds, average='macro', zero_division=0)
    
    return {
        'map3': map3_score,
        'top1_accuracy': top1_acc,
        'top3_accuracy': top3_acc,
        'f1_score': f1,
        'precision': prec,
        'recall': rec
    }

def rank_preds(pw, scores, orig_df):
    pw_temp = pw.copy()
    pw_temp['score'] = scores
    
    ranked = pw_temp.sort_values(['qid', 'score'], ascending=[True, False])
    grouped = ranked.groupby('qid')['option'].apply(lambda opts: list(opts)[:3]).to_dict()
    
    return [grouped.get(qid, ['A', 'B', 'C']) for qid in orig_df['id']]


In [9]:
# Cell 5: Pairwise Feature Engineering (Word Overlap + Jaccard + Length Ratio)
def make_pairwise_dataset(df, is_train=True):
    rows = []
    for _, r in df.iterrows():
        qid = r['id']
        prompt = str(r['prompt'])
        p_words = set(re.findall(r'\w+', prompt.lower()))
        ans = r.get('answer', None)
        
        for c in OPTION_COLS:
            opt_str = str(r[c])
            o_words = set(re.findall(r'\w+', opt_str.lower()))
            
            overlap = len(p_words.intersection(o_words))
            jaccard = overlap / max(len(p_words.union(o_words)), 1)
            len_ratio = len(opt_str) / max(len(prompt), 1)
            
            combined_text = f"{prompt} [SEP] {opt_str}"
            
            row = {
                'qid': qid,
                'option': c,
                'text': combined_text,
                'overlap': overlap,
                'jaccard': jaccard,
                'len_ratio': len_ratio
            }
            if is_train:
                row['label'] = 1 if (ans == c) else 0
            rows.append(row)
            
    return pd.DataFrame(rows)

pw_train = make_pairwise_dataset(train, is_train=True)
pw_test = make_pairwise_dataset(test, is_train=False)

print(f"Pairwise Train Shape: {pw_train.shape}")
print(f"Pairwise Test Shape : {pw_test.shape}")
pw_train.head(3)


Pairwise Train Shape: (10000, 7)
Pairwise Test Shape : (2500, 6)


,qid,option,text,overlap,jaccard,len_ratio,label
0,1,A,Pick the best possible answer: What is Martin ...,8,0.153846,2.161972,0
1,1,B,Pick the best possible answer: What is Martin ...,7,0.148936,1.816901,1
2,1,C,Pick the best possible answer: What is Martin ...,10,0.250000,1.563380,0


In [10]:
# Cell 6: TF-IDF Sparse Feature Matrix Construction
print("Fitting Word & Character TF-IDF Vectorizers...")

tfidf_word = TfidfVectorizer(max_features=80000, ngram_range=(1, 3), sublinear_tf=True, analyzer='word')
tfidf_char = TfidfVectorizer(max_features=50000, ngram_range=(2, 5), sublinear_tf=True, analyzer='char_wb')

combined_corpus = pd.concat([pw_train['text'], pw_test['text']])
tfidf_word.fit(combined_corpus)
tfidf_char.fit(combined_corpus)

Xw_tr = tfidf_word.transform(pw_train['text'])
Xw_te = tfidf_word.transform(pw_test['text'])

Xc_tr = tfidf_char.transform(pw_train['text'])
Xc_te = tfidf_char.transform(pw_test['text'])

X_num_tr = csr_matrix(pw_train[['overlap', 'jaccard', 'len_ratio']].values)
X_num_te = csr_matrix(pw_test[['overlap', 'jaccard', 'len_ratio']].values)

X_tr = hstack([Xw_tr, Xc_tr, X_num_tr])
X_te = hstack([Xw_te, Xc_te, X_num_te])

y_tr = pw_train['label'].values

print(f"Train Feature Matrix: {X_tr.shape}")
print(f"Test Feature Matrix : {X_te.shape}")


Fitting Word & Character TF-IDF Vectorizers...
Train Feature Matrix: (10000, 53432)
Test Feature Matrix : (2500, 53432)


In [11]:
# Cell 7: Fold grouping key (stem vs qid)
# Prompts reuse a small set of framing phrases. Stripping them collapses paraphrases
# of the same underlying question to a shared stem. Grouping folds by stem keeps every
# paraphrase of a question inside one fold; grouping by qid does not, and lets the same
# question appear on both sides of the split. Run the notebook once with each setting
# to measure how much of the validation score comes from that overlap.

GROUP_BY = 'qid'           # 'stem' for the honest split, 'qid' for the leaky one

PREFIX_PATTERNS = [
    r'^\s*pick the best possible answer\s*[:\-]?\s*',
    r'^\s*select the most accurate option\s*[:\-]?\s*',
    r'^\s*determine the correct option\s*[:\-]?\s*',
    r'^\s*identify the correct statement\s*[:\-]?\s*',
    r'^\s*choose the correct answer\s*[:\-]?\s*',
    r'^\s*which of the following is correct\??\s*',
]
SUFFIX_PATTERNS = [
    r'\s+based on the given context\.?\s*$',
    r'\s+based on the given information\.?\s*$',
    r'\s+from the following choices\.?\s*$',
    r'\s+among the listed options\.?\s*$',
    r'\s+carefully\.?\s*$',
    r'\s+above\.?\s*$',
]

def strip_framing(text):
    text = str(text)
    prev = None
    while prev != text:
        prev = text
        for p in PREFIX_PATTERNS:
            text = re.sub(p, '', text, flags=re.IGNORECASE)
        for p in SUFFIX_PATTERNS:
            text = re.sub(p, '', text, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', text).strip().lower()

train['stem'] = train['prompt'].apply(strip_framing)
qid_to_stem   = dict(zip(train['id'], train['stem']))

if GROUP_BY == 'stem':
    groups = pw_train['qid'].map(qid_to_stem).values
else:
    groups = pw_train['qid'].values

print(f"unique prompts: {train['prompt'].nunique()} | unique stems: {train['stem'].nunique()}")
print(f"grouping folds by: {GROUP_BY} | distinct groups: {len(set(groups))}")


unique prompts: 1758 | unique stems: 252
grouping folds by: qid | distinct groups: 2000


In [12]:
# Cell 8: 5-Fold Training Loop with Detailed Fold-Wise & Model-Wise Wandb Logging
# C was 6.0, chosen before the fold grouping was corrected. Under stem-grouped folds
# the sweep in Cell 10 peaks at 0.03, and 6.0 sits near the worst end of the grid: the
# earlier value had been tuned against the leakage rather than against the task.
C_LR = 0.03

gkf = GroupKFold(n_splits=5)

oof_lr = np.zeros(len(pw_train))
oof_ridge = np.zeros(len(pw_train))
oof_sgd = np.zeros(len(pw_train))

tst_lr = np.zeros(len(pw_test))
tst_ridge = np.zeros(len(pw_test))
tst_sgd = np.zeros(len(pw_test))

config_dict = {
    "architecture": "Logistic Regression + Ridge + SGD Ensemble",
    "dataset": "Smart MCQ Solver Challenge",
    "C_lr": C_LR,
    "C_lr_previous": 6.0,
    "ridge_output": "expit(decision_function)",
    "alpha_ridge": 1.0,
    "loss_sgd": "log_loss",
    "max_features_word": 80000,
    "max_features_char": 50000,
    "ngram_range_word": "(1,3)",
    "ngram_range_char": "(2,5)",
    "n_splits": 5,
    "group_by": GROUP_BY,
    "seed": SEED,
    "ensemble_weights": {"LR": 0.50, "Ridge": 0.30, "SGD": 0.20}
}

if USE_WANDB:
    run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name=f"LR-TFIDF-{GROUP_BY}",
        config=config_dict
    )
    wandb.define_metric("epoch")
    wandb.define_metric("train/*", step_metric="epoch")
    wandb.define_metric("val/*",   step_metric="epoch")
    wandb.define_metric("fold")
    wandb.define_metric("fold/*",  step_metric="fold")

print("\nStarting 5-Fold Cross Validation...")

fold_metrics_list = []

for fold, (ti, vi) in enumerate(gkf.split(X_tr, y_tr, groups)):
    print(f"\n--- Fold {fold+1}/5 ---")
    
    # 1. Logistic Regression
    lr = LogisticRegression(C=C_LR, max_iter=2000)
    lr.fit(X_tr[ti], y_tr[ti])
    oof_lr[vi] = lr.predict_proba(X_tr[vi])[:, 1]
    tst_lr += lr.predict_proba(X_te)[:, 1] / 5.0
    
    # 2. Ridge Classifier
    ridge = RidgeClassifier(alpha=1.0)
    ridge.fit(X_tr[ti], y_tr[ti])
    # RidgeClassifier.decision_function returns an unbounded margin, not a probability.
    # Blended raw against two probability vectors it dominates by scale, which is what
    # dragged the earlier blend below its own SGD member. expit maps it to (0, 1) so the
    # three scores are on a common scale before the weighted sum.
    oof_ridge[vi] = expit(ridge.decision_function(X_tr[vi]))
    tst_ridge += expit(ridge.decision_function(X_te)) / 5.0
    
    # 3. SGD Classifier
    sgd = SGDClassifier(loss='log_loss', max_iter=1000, random_state=SEED)
    sgd.fit(X_tr[ti], y_tr[ti])
    oof_sgd[vi] = sgd.predict_proba(X_tr[vi])[:, 1]
    tst_sgd += sgd.predict_proba(X_te)[:, 1] / 5.0
    
    # Fold Blend Predictions
    fold_blend_scores = 0.50 * oof_lr[vi] + 0.30 * oof_ridge[vi] + 0.20 * oof_sgd[vi]
    
    val_qids = pw_train.iloc[vi]['qid'].unique()
    val_df_orig = train[train['id'].isin(val_qids)].reset_index(drop=True)
    val_pw_df = pw_train[pw_train['qid'].isin(val_qids)].reset_index(drop=True)
    
    fold_preds = rank_preds(val_pw_df, fold_blend_scores, val_df_orig)
    fold_actuals = val_df_orig['answer'].tolist()
    
    # Compute Fold Metrics
    m = compute_all_metrics(fold_actuals, fold_preds)
    fold_val_loss = log_loss(val_pw_df['label'], fold_blend_scores)
    
    print(f" Fold {fold+1} Loss: {fold_val_loss:.4f} | MAP@3: {m['map3']:.4f} | Acc: {m['top1_accuracy']:.4f} | F1: {m['f1_score']:.4f}")
    
    fold_metrics_list.append(m)
    
    # Log Fold Metrics to Wandb
    if USE_WANDB:
        wandb.log({
            "fold": fold + 1,
            "fold/val_loss": fold_val_loss,
            "fold/val_map3": m['map3'],
            "fold/val_top1_accuracy": m['top1_accuracy'],
            "fold/val_top3_accuracy": m['top3_accuracy'],
            "fold/val_f1_score": m['f1_score'],
            "fold/val_precision": m['precision'],
            "fold/val_recall": m['recall'],
            "fold/val_map3_lr":    mapk(fold_actuals, rank_preds(val_pw_df, oof_lr[vi], val_df_orig)),
            "fold/val_map3_ridge": mapk(fold_actuals, rank_preds(val_pw_df, oof_ridge[vi], val_df_orig)),
            "fold/val_map3_sgd":   mapk(fold_actuals, rank_preds(val_pw_df, oof_sgd[vi], val_df_orig)),
        })

fold_map3 = np.array([m['map3'] for m in fold_metrics_list])
print(f"\nfold MAP@3: mean={fold_map3.mean():.4f} sd={fold_map3.std():.4f} "
      f"min={fold_map3.min():.4f} max={fold_map3.max():.4f}")
if USE_WANDB:
    wandb.summary["fold_map3_mean"] = float(fold_map3.mean())
    wandb.summary["fold_map3_std"]  = float(fold_map3.std())
    wandb.summary["fold_map3_min"]  = float(fold_map3.min())
    wandb.summary["fold_map3_max"]  = float(fold_map3.max())

# Overall Out-Of-Fold Predictions
oof_blend = 0.50 * oof_lr + 0.30 * oof_ridge + 0.20 * oof_sgd
tst_blend = 0.50 * tst_lr + 0.30 * tst_ridge + 0.20 * tst_sgd

overall_actuals = train['answer'].tolist()

# Overall Model Metrics
m_lr = compute_all_metrics(overall_actuals, rank_preds(pw_train, oof_lr, train))
m_ridge = compute_all_metrics(overall_actuals, rank_preds(pw_train, oof_ridge, train))
m_sgd = compute_all_metrics(overall_actuals, rank_preds(pw_train, oof_sgd, train))
m_blend = compute_all_metrics(overall_actuals, rank_preds(pw_train, oof_blend, train))

oof_loss = log_loss(pw_train['label'], oof_blend)

print("\n==================================================")
print("       OVERALL OUT-OF-FOLD PERFORMANCE SUMMARY")
print("==================================================")
print(f" Primary Kaggle Metric (MAP@3): {m_blend['map3']:.4f}")
print(f" Top-1 Accuracy               : {m_blend['top1_accuracy']:.4f}")
print(f" Top-3 Recall Accuracy        : {m_blend['top3_accuracy']:.4f}")
print(f" Macro F1 Score               : {m_blend['f1_score']:.4f}")
print(f" Macro Precision              : {m_blend['precision']:.4f}")
print(f" Macro Recall                 : {m_blend['recall']:.4f}")
print(f" Pairwise Log Loss            : {oof_loss:.4f}")
print("==================================================\n")

# Log Overall Summary & Comparative Table to Wandb
if USE_WANDB:
    wandb.log({
        "oof/overall_map3": m_blend['map3'],
        "oof/overall_top1_accuracy": m_blend['top1_accuracy'],
        "oof/overall_top3_accuracy": m_blend['top3_accuracy'],
        "oof/overall_f1_score": m_blend['f1_score'],
        "oof/overall_precision": m_blend['precision'],
        "oof/overall_recall": m_blend['recall'],
        "oof/overall_val_loss": oof_loss,
        "models/lr_map3": m_lr['map3'],
        "models/ridge_map3": m_ridge['map3'],
        "models/sgd_map3": m_sgd['map3'],
    })
    
    # Create Wandb Summary Table
    table = wandb.Table(columns=["Model Architecture", "MAP@3 Score", "Top-1 Accuracy", "Top-3 Accuracy", "F1 Score", "Precision", "Recall"])
    table.add_data(f"Logistic Regression (C={C_LR}, folds by {GROUP_BY})", m_lr['map3'], m_lr['top1_accuracy'], m_lr['top3_accuracy'], m_lr['f1_score'], m_lr['precision'], m_lr['recall'])
    table.add_data(f"Ridge Classifier (alpha=1.0, folds by {GROUP_BY})", m_ridge['map3'], m_ridge['top1_accuracy'], m_ridge['top3_accuracy'], m_ridge['f1_score'], m_ridge['precision'], m_ridge['recall'])
    table.add_data(f"SGD Classifier (log_loss, folds by {GROUP_BY})", m_sgd['map3'], m_sgd['top1_accuracy'], m_sgd['top3_accuracy'], m_sgd['f1_score'], m_sgd['precision'], m_sgd['recall'])
    table.add_data(f"Ensemble Blend (LR+Ridge+SGD, folds by {GROUP_BY})", m_blend['map3'], m_blend['top1_accuracy'], m_blend['top3_accuracy'], m_blend['f1_score'], m_blend['precision'], m_blend['recall'])
    
    wandb.log({"model_comparative_summary": table})



Starting 5-Fold Cross Validation...

--- Fold 1/5 ---
 Fold 1 Loss: 0.3761 | MAP@3: 1.0000 | Acc: 1.0000 | F1: 1.0000

--- Fold 2/5 ---
 Fold 2 Loss: 0.3828 | MAP@3: 0.9950 | Acc: 0.9900 | F1: 0.9899

--- Fold 3/5 ---
 Fold 3 Loss: 0.3760 | MAP@3: 0.9962 | Acc: 0.9925 | F1: 0.9928

--- Fold 4/5 ---
 Fold 4 Loss: 0.3750 | MAP@3: 0.9954 | Acc: 0.9925 | F1: 0.9919

--- Fold 5/5 ---
 Fold 5 Loss: 0.3767 | MAP@3: 0.9962 | Acc: 0.9925 | F1: 0.9926

fold MAP@3: mean=0.9966 sd=0.0018 min=0.9950 max=1.0000

       OVERALL OUT-OF-FOLD PERFORMANCE SUMMARY
 Primary Kaggle Metric (MAP@3): 0.9966
 Top-1 Accuracy               : 0.9935
 Top-3 Recall Accuracy        : 1.0000
 Macro F1 Score               : 0.9934
 Macro Precision              : 0.9941
 Macro Recall                 : 0.9929
 Pairwise Log Loss            : 0.3773



In [13]:
# Cell 9: Epoch-wise training curves via stochastic gradient descent
# LogisticRegression is fitted by L-BFGS, which exposes no per-iteration objective, so it
# cannot produce a loss curve. SGDClassifier(loss='log_loss') optimises the identical
# logistic objective one pass at a time, so calling partial_fit in a loop gives a genuine
# per-epoch trace of training loss, validation loss and the ranking metrics. alpha=1e-3 is
# the L2 strength closest to the optimum found by the regularisation sweep below.
# About 10 seconds for 15 epochs across 5 folds.

SGD_EPOCHS = 15
SGD_ALPHA  = 1e-3

if USE_WANDB:
    wandb.define_metric("epoch")
    wandb.define_metric("train/*", step_metric="epoch")
    wandb.define_metric("val/*",   step_metric="epoch")

curve_rows = []
for fold, (ti, vi) in enumerate(gkf.split(X_tr, y_tr, groups)):
    clf = SGDClassifier(loss='log_loss', alpha=SGD_ALPHA,
                        learning_rate='optimal', random_state=SEED)

    val_pw_df    = pw_train.iloc[vi].reset_index(drop=True)
    val_df_orig  = train[train['id'].isin(val_pw_df['qid'].unique())].reset_index(drop=True)
    fold_actuals = val_df_orig['answer'].tolist()

    for epoch in range(1, SGD_EPOCHS + 1):
        clf.partial_fit(X_tr[ti], y_tr[ti], classes=np.array([0, 1]))

        p_tr = clf.predict_proba(X_tr[ti])[:, 1]
        p_va = clf.predict_proba(X_tr[vi])[:, 1]
        tr_loss = log_loss(y_tr[ti], np.clip(p_tr, 1e-6, 1 - 1e-6))
        va_loss = log_loss(y_tr[vi], np.clip(p_va, 1e-6, 1 - 1e-6))
        m_ep    = compute_all_metrics(fold_actuals,
                                      rank_preds(val_pw_df, p_va, val_df_orig))

        curve_rows.append({"fold": fold + 1, "epoch": epoch,
                           "train_loss": tr_loss, "val_loss": va_loss, **m_ep})

        if USE_WANDB:
            wandb.log({
                "epoch":             epoch,
                "fold":              fold + 1,
                "train/loss":        tr_loss,
                "val/loss":          va_loss,
                "val/map3":          m_ep['map3'],
                "val/top1_accuracy": m_ep['top1_accuracy'],
                "val/top3_accuracy": m_ep['top3_accuracy'],
                "val/f1_score":      m_ep['f1_score'],
                "val/precision":     m_ep['precision'],
                "val/recall":        m_ep['recall'],
            })

    print(f"  fold {fold+1} | final train_loss={tr_loss:.4f} val_loss={va_loss:.4f} "
          f"MAP@3={m_ep['map3']:.4f}")

curve_df = pd.DataFrame(curve_rows)
print()
print(curve_df.groupby('epoch')[['train_loss', 'val_loss', 'map3', 'top1_accuracy']]
      .mean().round(4).to_markdown())


  fold 1 | final train_loss=0.4253 val_loss=0.4325 MAP@3=0.9646
  fold 2 | final train_loss=0.4338 val_loss=0.4370 MAP@3=0.9337
  fold 3 | final train_loss=0.4241 val_loss=0.4311 MAP@3=0.9717
  fold 4 | final train_loss=0.4402 val_loss=0.4499 MAP@3=0.9321
  fold 5 | final train_loss=0.4208 val_loss=0.4303 MAP@3=0.9354

|   epoch |   train_loss |   val_loss |   map3 |   top1_accuracy |
|--------:|-------------:|-----------:|-------:|----------------:|
|       1 |       1.1572 |     1.2049 | 0.7907 |          0.7015 |
|       2 |       0.6767 |     0.6941 | 0.8626 |          0.794  |
|       3 |       0.5491 |     0.5595 | 0.9004 |          0.8515 |
|       4 |       0.4999 |     0.5086 | 0.9184 |          0.88   |
|       5 |       0.4753 |     0.4834 | 0.9257 |          0.893  |
|       6 |       0.4607 |     0.4686 | 0.9299 |          0.898  |
|       7 |       0.4513 |     0.459  | 0.9368 |          0.909  |
|       8 |       0.4447 |     0.4524 | 0.9401 |          0.913  |
|       9

In [14]:
# Cell 10: Regularisation sweep
# C is the inverse regularisation strength, so a small C means a heavier penalty. On a
# 53k-feature matrix built from 252 distinct questions the optimum sits far below C = 1,
# so the grid spans four orders of magnitude and is read on a log axis.
# About 9 x 5 = 45 fits, roughly 3 minutes.

RUN_C_SWEEP = True
C_GRID = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]

if RUN_C_SWEEP:
    if USE_WANDB:
        wandb.define_metric("sweep/step")
        wandb.define_metric("sweep/*", step_metric="sweep/step")

    sweep_rows = []
    for i, C in enumerate(C_GRID):
        oof_c = np.zeros(len(pw_train))
        for ti, vi in gkf.split(X_tr, y_tr, groups):
            clf = LogisticRegression(C=C, max_iter=2000)
            clf.fit(X_tr[ti], y_tr[ti])
            oof_c[vi] = clf.predict_proba(X_tr[vi])[:, 1]

        m_c = compute_all_metrics(overall_actuals, rank_preds(pw_train, oof_c, train))
        sweep_rows.append({"C": C, "log10_C": float(np.log10(C)), **m_c})
        print(f"  C={C:<8} MAP@3={m_c['map3']:.4f} | top1={m_c['top1_accuracy']:.4f} "
              f"| top3={m_c['top3_accuracy']:.4f}")

        if USE_WANDB:
            wandb.log({
                "sweep/step":     i,
                "sweep/C":        C,
                "sweep/log10_C":  float(np.log10(C)),
                "sweep/oof_map3": m_c['map3'],
                "sweep/oof_top1": m_c['top1_accuracy'],
                "sweep/oof_top3": m_c['top3_accuracy'],
                "sweep/oof_f1":   m_c['f1_score'],
            })

    sweep_df = pd.DataFrame(sweep_rows)
    best = sweep_df.loc[sweep_df['map3'].idxmax()]
    print(f"\nbest C = {best['C']} at MAP@3 = {best['map3']:.4f}\n")
    print(sweep_df.round(4).to_markdown(index=False))


  C=0.001    MAP@3=0.6621 | top1=0.5070 | top3=0.8675
  C=0.003    MAP@3=0.7491 | top1=0.6135 | top3=0.9230
  C=0.01     MAP@3=0.8774 | top1=0.7975 | top3=0.9640
  C=0.03     MAP@3=0.9350 | top1=0.8905 | top3=0.9815
  C=0.1      MAP@3=0.9653 | top1=0.9440 | top3=0.9885
  C=0.3      MAP@3=0.9828 | top1=0.9705 | top3=0.9965
  C=1.0      MAP@3=0.9932 | top1=0.9875 | top3=0.9990
  C=3.0      MAP@3=0.9963 | top1=0.9930 | top3=1.0000
  C=10.0     MAP@3=0.9974 | top1=0.9950 | top3=1.0000

best C = 10.0 at MAP@3 = 0.9974

|      C |   log10_C |   map3 |   top1_accuracy |   top3_accuracy |   f1_score |   precision |   recall |
|-------:|----------:|-------:|----------------:|----------------:|-----------:|------------:|---------:|
|  0.001 |   -3      | 0.6621 |          0.507  |          0.8675 |     0.5021 |      0.5031 |   0.5072 |
|  0.003 |   -2.5229 | 0.7491 |          0.6135 |          0.923  |     0.6084 |      0.6111 |   0.6122 |
|  0.01  |   -2      | 0.8774 |          0.7975 |       

In [15]:
# Cell 11: Learning curve over training-set size
# Train on an increasing share of the available stems and score on stems the model has
# never seen. This separates two explanations for the ceiling: too few labelled questions,
# or a dataset whose duplication and contradictory labels cap what is learnable.

RUN_LEARNING_CURVE = True
FRACTIONS = [0.2, 0.4, 0.6, 0.8, 1.0]
BEST_C    = 0.03

if RUN_LEARNING_CURVE:
    if USE_WANDB:
        wandb.define_metric("lc/train_stems")
        wandb.define_metric("lc/*", step_metric="lc/train_stems")

    rng      = np.random.default_rng(SEED)
    row_stem = pw_train['qid'].map(qid_to_stem).values

    lc_rows = []
    for frac in FRACTIONS:
        fold_scores, n_stems_used = [], []
        for ti, vi in gkf.split(X_tr, y_tr, groups):
            train_stems = np.array(sorted(set(row_stem[ti])))
            n_keep      = max(1, int(round(frac * len(train_stems))))
            keep        = set(rng.choice(train_stems, size=n_keep, replace=False))
            ti_sub      = ti[np.isin(row_stem[ti], list(keep))]
            n_stems_used.append(n_keep)

            clf = LogisticRegression(C=BEST_C, max_iter=2000)
            clf.fit(X_tr[ti_sub], y_tr[ti_sub])
            scores = clf.predict_proba(X_tr[vi])[:, 1]

            val_pw   = pw_train.iloc[vi].reset_index(drop=True)
            val_orig = train[train['id'].isin(val_pw['qid'].unique())].reset_index(drop=True)
            fold_scores.append(mapk(val_orig['answer'].tolist(),
                                    rank_preds(val_pw, scores, val_orig)))

        mean_stems = int(round(np.mean(n_stems_used)))
        lc_rows.append({"fraction": frac, "train_stems": mean_stems,
                        "map3_mean": float(np.mean(fold_scores)),
                        "map3_std":  float(np.std(fold_scores))})
        print(f"  {frac:.0%} of stems (~{mean_stems:3d}) | "
              f"MAP@3 = {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}")

        if USE_WANDB:
            wandb.log({"lc/train_stems": mean_stems,
                       "lc/map3_mean":   float(np.mean(fold_scores)),
                       "lc/map3_std":    float(np.std(fold_scores))})

    lc_df = pd.DataFrame(lc_rows)
    print()
    print(lc_df.round(4).to_markdown(index=False))


  20% of stems (~ 49) | MAP@3 = 0.6220 +/- 0.0244
  40% of stems (~ 98) | MAP@3 = 0.7172 +/- 0.0278
  60% of stems (~147) | MAP@3 = 0.7932 +/- 0.0331
  80% of stems (~196) | MAP@3 = 0.8824 +/- 0.0128
  100% of stems (~245) | MAP@3 = 0.9350 +/- 0.0119

|   fraction |   train_stems |   map3_mean |   map3_std |
|-----------:|--------------:|------------:|-----------:|
|        0.2 |            49 |      0.622  |     0.0244 |
|        0.4 |            98 |      0.7172 |     0.0278 |
|        0.6 |           147 |      0.7932 |     0.0331 |
|        0.8 |           196 |      0.8824 |     0.0128 |
|        1   |           245 |      0.935  |     0.0119 |


In [16]:
# Cell 12: Final Submission Generation & Comparative Report Table
test_preds = rank_preds(pw_test, tst_blend, test)

submission = pd.DataFrame({
    'ID': test['id'],
    'Prediction': [' '.join(p) for p in test_preds]
})

submission.to_csv('submission.csv', index=False)
submission.to_csv(f'preds_lr_{GROUP_BY}.csv', index=False)
print("Successfully generated submission.csv!")

assert len(submission) == len(test), f"Expected {len(test)} rows, got {len(submission)}"
assert submission['Prediction'].str.match(r'^[A-E] [A-E] [A-E]$').all(), "Invalid prediction format!"

# Print Comparative Summary Markdown Table for Project Report
report_df = pd.DataFrame([
    {"Model Architecture": f"Logistic Regression (C={C_LR})", "MAP@3": f"{m_lr['map3']:.4f}", "Top-1 Acc": f"{m_lr['top1_accuracy']:.4f}", "Top-3 Acc": f"{m_lr['top3_accuracy']:.4f}", "F1 Score": f"{m_lr['f1_score']:.4f}"},
    {"Model Architecture": "Ridge Classifier (alpha=1.0)", "MAP@3": f"{m_ridge['map3']:.4f}", "Top-1 Acc": f"{m_ridge['top1_accuracy']:.4f}", "Top-3 Acc": f"{m_ridge['top3_accuracy']:.4f}", "F1 Score": f"{m_ridge['f1_score']:.4f}"},
    {"Model Architecture": "SGD Classifier (log_loss)", "MAP@3": f"{m_sgd['map3']:.4f}", "Top-1 Acc": f"{m_sgd['top1_accuracy']:.4f}", "Top-3 Acc": f"{m_sgd['top3_accuracy']:.4f}", "F1 Score": f"{m_sgd['f1_score']:.4f}"},
    {"Model Architecture": "Ensemble Blend (LR+Ridge+SGD)", "MAP@3": f"{m_blend['map3']:.4f}", "Top-1 Acc": f"{m_blend['top1_accuracy']:.4f}", "Top-3 Acc": f"{m_blend['top3_accuracy']:.4f}", "F1 Score": f"{m_blend['f1_score']:.4f}"}
])

print("\n" + "="*70)
print("          MODEL COMPARATIVE REPORT FOR REPORT DOCUMENTATION")
print("="*70)
print(report_df.to_string(index=False))
print("="*70 + "\n")

print("Submission Sample:")
submission.head(10)

# ----- Blend against its own members -----
# The weighted blend is only worth reporting if it beats every member it is built from.
members = {"Logistic Regression": oof_lr, "Ridge (calibrated)": oof_ridge,
           "SGD (log loss)": oof_sgd,
           "Blend 0.50/0.30/0.20": 0.50 * oof_lr + 0.30 * oof_ridge + 0.20 * oof_sgd}
rows = []
for nm, sc in members.items():
    mm = compute_all_metrics(overall_actuals, rank_preds(pw_train, sc, train))
    rows.append({"model": nm, **mm})
member_df = pd.DataFrame(rows)
print("\n--- OOF by ensemble member ---")
print(member_df.round(4).to_markdown(index=False))

best_member = member_df.iloc[member_df["map3"].idxmax()]
print(f"\nbest: {best_member['model']} at MAP@3 = {best_member['map3']:.4f}")

# Close the W&B run. This must be the last wandb call in the notebook: cells 8, 9, 10
# and 11 all log to the same run, so finishing any earlier would discard everything
# logged after it and leave the run open in the dashboard.
if USE_WANDB:
    wandb.finish()


Successfully generated submission.csv!

          MODEL COMPARATIVE REPORT FOR REPORT DOCUMENTATION
           Model Architecture  MAP@3 Top-1 Acc Top-3 Acc F1 Score
 Logistic Regression (C=0.03) 0.9350    0.8905    0.9815   0.8885
 Ridge Classifier (alpha=1.0) 0.9974    0.9950    1.0000   0.9949
    SGD Classifier (log_loss) 0.9932    0.9875    0.9990   0.9875
Ensemble Blend (LR+Ridge+SGD) 0.9966    0.9935    1.0000   0.9934

Submission Sample:

--- OOF by ensemble member ---
| model                |   map3 |   top1_accuracy |   top3_accuracy |   f1_score |   precision |   recall |
|:---------------------|-------:|----------------:|----------------:|-----------:|------------:|---------:|
| Logistic Regression  | 0.935  |          0.8905 |          0.9815 |     0.8885 |      0.888  |   0.891  |
| Ridge (calibrated)   | 0.9974 |          0.995  |          1      |     0.9949 |      0.9956 |   0.9944 |
| SGD (log loss)       | 0.9932 |          0.9875 |          0.999  |     0.9875 |    

epoch,▂▃▃▄▅▅▇▇█▁▃▃▄▅▅▇█▁▁▃▅▅▇▇█▂▃▃▄▅▅▇▇▁▁▃▅▅▆█
fold,▁▅▆█▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▅▅▅▅▅▅▆▆▆▆▆▆▆▆██████
fold/val_f1_score,█▁▃▂▃
fold/val_loss,▂█▂▁▃
fold/val_map3,█▁▃▂▃
fold/val_map3_lr,█▅█▂▁
fold/val_map3_ridge,█▁▃▄▆
fold/val_map3_sgd,█▃▆▅▁
fold/val_precision,█▁▂▁▁
fold/val_recall,█▁▄▃▄
+30,...
